# Swift: Neural Video Streaming Pipeline

## 1. Environment Setup

In [ ]:
import os
import sys

os.makedirs("data", exist_ok=True)
os.makedirs("data/original_frames", exist_ok=True)
os.makedirs("data/videos", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("outputs/evaluation", exist_ok=True)
os.makedirs("configs", exist_ok=True)

print("Project root:", os.getcwd())
print("Python version:", sys.version)

### Install the python libraries

In [ ]:
%pip install -r requirements.txt

## 2. Hardware Profiling

In [ ]:
!python scripts/profile_hardware.py

In [ ]:
import torch

if torch.cuda.is_available():
  print("CUDA (GPU) is available! Device name:", torch.cuda.get_device_name(0))
  print("Number of GPUs available:", torch.cuda.device_count())
else:
  print("CUDA (GPU) is NOT available. Training will proceed on CPU.")

## 3. Extracting frames from V1.mp4
Add youtube video link in the placeholder "YT_Video"

In [ ]:
!yt-dlp -f "bestvideo[height<=1080]+bestaudio/best[height<=1080]" -o "data/videos/V1" --merge-output-format mp4 "YT_Video"

In [ ]:
!python scripts/extract_frames.py --video data/videos/V1.mp4 --frame-skip 2 --output-dir data/original_frames --backend ffmpeg --max-frames 500

## 4. Training Stage 1: Multi-Level Autoencoder

In [ ]:
!python -m codec.autoencoder.train_script --train-dir data/original_frames --epochs 20 --batch-size 4 --save-dir models/autoencoder

## 5. Training Stage 2: Singleshot Decoder

In [ ]:
!python -m codec.singleshot.train_singleshot

## 5. End-to-End Evaluation
Run the full evaluation suite including PSNR, SSIM, Latency, and Baseline comparisons.

In [ ]:
!python scripts/evaluate.py

## 6. Visualize Results
Show the generated Rate-Distortion curves and error heatmaps.

In [ ]:
from IPython.display import Image, display

print("\n--- RD-Curve Comparison ---")
rd_curve_path = "outputs/evaluation/rd_curve_comparison.png"
if os.path.exists(rd_curve_path):
    display(Image(filename=rd_curve_path))
else:
    print("RD Curve not found. Ensure evaluate.py ran successfully.")

print("\n--- Sample Pixel Error Heatmap ---")
heatmap_dir = "outputs/evaluation/heatmaps"
if os.path.exists(heatmap_dir):
    heatmaps = [f for f in os.listdir(heatmap_dir) if f.endswith('.png')]
    if heatmaps:
        display(Image(filename=os.path.join(heatmap_dir, heatmaps[0])))
else:
    print("Heatmaps not found.")